In [1]:
!git clone https://github.com/Angela-Benedict/ds2002-fa26.git

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

import os

for root, dirs, files in os.walk('/content'):
    for file in files:
        if file.endswith('.db'):
            print(os.path.join(root, file))

fatal: destination path 'ds2002-fa26' already exists and is not an empty directory.
/content/inventory_and_sales (1).db
/content/inventory_and_sales.db
/content/.config/default_configs.db
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/ds2002-fa26/notebooks/06-midterm-walmart/data/inventory_and_sales.db
/content/ds2002-fa26/notebooks/walmart-hurricane-project/data/inventory_and_sales.db


In [2]:
conn = sqlite3.connect(
    '/content/ds2002-fa26/notebooks/walmart-hurricane-project/data/inventory_and_sales.db'
)

In [3]:
tables = pd.read_sql_query("""
    SELECT name
    FROM sqlite_master
    WHERE type='table';
""", conn)

print(tables)

                  name
0  daily_sales_summary
1     inventory_levels


In [4]:
inventory = pd.read_sql_query(
    "SELECT * FROM inventory_levels",
    conn
)

game_skus = [
    'BG-MONO',
    'GAME-MONO',
    'MONOPOLY-01',
    'CARDS-01',
    'PLAYCARD-STD'
]

game_inventory = inventory[
    inventory['sku'].isin(game_skus)
]

print(game_inventory.groupby('sku').agg(
    stores=('store_id', 'nunique'),
    average_inventory=('units_on_hand', 'mean'),
    total_on_order=('units_on_order', 'sum')
))

              stores  average_inventory  total_on_order
sku                                                    
BG-MONO           12          77.709677          3967.0
CARDS-01          12          80.072917          4075.0
GAME-MONO         12          71.967391          3903.0
MONOPOLY-01       12          88.945652          3809.0
PLAYCARD-STD      12          81.534247          2804.0


In [5]:
# Load sales data
sales = pd.read_sql_query(
    "SELECT * FROM daily_sales_summary",
    conn
)

# Load inventory data
inventory = pd.read_sql_query(
    "SELECT * FROM inventory_levels",
    conn
)

# Check the data
print("Sales data:")
display(sales.head())

print("Inventory data:")
display(inventory.head())

Sales data:


,date,store_id,category,total_units,total_revenue
0,2024-09-12,FL-239,Emergency,90.0,1582.55
1,2024-09-12,FL-239,Breakfast,57.0,356.99
2,2024-09-12,FL-239,Alcohol,41.0,246.65
3,2024-09-12,FL-239,Entertainment,53.0,916.56
4,2024-09-12,FL-239,Grocery,69.0,339.30


Inventory data:


,date,store_id,sku,units_on_hand,units_on_order,reorder_point
0,2024-09-12,FL-239,CHIPS-VP,112,5.0,31
1,2024-09-12,FL-239,PAPERTOWEL6,106,46.0,36
2,2024-09-12,FL-239,PT-12,62,36.0,30
3,2024-09-12,FL-239,PORTCHARGER,185,62.0,40
4,2024-09-12,FL-239,BFORM-01,88,1.0,35


In [6]:
entertainment = sales[
    sales['category'] == 'Entertainment'
]

display(entertainment.head(10))

,date,store_id,category,total_units,total_revenue
3,2024-09-12,FL-239,Entertainment,53.0,916.56
13,2024-09-12,FL-105,Entertainment,62.0,1081.91
23,20240912,FL-330,Entertainment,21.0,192.86
33,2024-09-12,FL-412,Entertainment,26.0,445.07
43,2024-09-12,FL-287,Entertainment,66.0,933.55
53,2024-09-12,FL-156,Entertainment,49.0,623.30
63,2024-09-12,FL-501,Entertainment,77.0,1317.32
73,2024-09-12,FL-078,Entertainment,77.0,1091.91
83,2024-09-12,FL-619,Entertainment,46.0,256.16
93,20240912,FL-445,Entertainment,54.0,883.90


In [7]:
# Convert dates to datetime
# format='mixed' handles the two date formats in your database
entertainment['date'] = pd.to_datetime(
    entertainment['date'],
    format='mixed'
)

# Combine sales across all stores by date
daily_sales = entertainment.groupby('date').agg(
    total_units=('total_units', 'sum'),
    total_revenue=('total_revenue', 'sum')
).reset_index()

display(daily_sales)

/tmp/ipykernel_10815/2956138127.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  entertainment['date'] = pd.to_datetime(


,date,total_units,total_revenue
0,2024-09-01,544.0,6166.78
1,2024-09-02,498.0,5671.31
2,2024-09-03,504.0,5356.50
3,2024-09-04,600.5,8603.30
4,2024-09-05,551.0,7673.41
5,2024-09-06,464.0,6028.19
6,2024-09-07,624.0,6268.56
7,2024-09-08,622.0,5835.79
8,2024-09-09,613.0,8135.51
9,2024-09-10,576.0,6054.80


In [8]:
periods = {
    'Baseline': ('2024-09-08', '2024-09-15'),
    'Surge': ('2024-09-19', '2024-09-25')
}

for period, (start, end) in periods.items():

    data = daily_sales[
        (daily_sales['date'] >= start) &
        (daily_sales['date'] <= end)
    ]

    print("\n", period)
    print("Number of days:", len(data))
    print("Average daily units:",
          round(data['total_units'].mean(), 2))
    print("Average daily revenue:",
          round(data['total_revenue'].mean(), 2))


 Baseline
Number of days: 8
Average daily units: 592.69
Average daily revenue: 6964.86

 Surge
Number of days: 7
Average daily units: 1394.5
Average daily revenue: 16237.89


In [9]:
# Calculate percentage increase in units sold
units_increase = (
    (1394.50 - 592.69) / 592.69
) * 100

# Calculate percentage increase in revenue
revenue_increase = (
    (16237.89 - 6964.86) / 6964.86
) * 100

print("Percentage increase in units:",
      round(units_increase, 1), "%")

print("Percentage increase in revenue:",
      round(revenue_increase, 1), "%")

Percentage increase in units: 135.3 %
Percentage increase in revenue: 133.1 %


- Used AI to help me write this code...check the numbers and also the time windows. I also am unable to finish this tonight I'm sorry. We have to still finish the SKU consolidation and then provide an analysis about the board games...I will try to work on this tomorrow night. THANKS GUYS.

In [10]:
game_skus = [
    'BG-MONO',
    'GAME-MONO',
    'MONOPOLY-01',
    'CARDS-01',
    'PLAYCARD-STD'
]

game_inventory = inventory[
    inventory['sku'].isin(game_skus)
].copy()

display(game_inventory.head(10))

,date,store_id,sku,units_on_hand,units_on_order,reorder_point
26,2024-09-12,FL-330,MONOPOLY-01,178,39.0,45
30,2024-09-12,FL-330,BG-MONO,86,38.0,42
44,2024-09-12,FL-412,PLAYCARD-STD,52,8.0,29
52,2024-09-12,FL-287,CARDS-01,142,94.0,22
60,2024-09-12,FL-156,PLAYCARD-STD,181,26.0,45
65,2024-09-12,FL-156,BG-MONO,81,21.0,50
69,2024-09-12,FL-156,CARDS-01,141,25.0,42
77,2024-09-12,FL-501,MONOPOLY-01,171,10.0,32
80,2024-09-12,FL-501,BG-MONO,191,57.0,30
107,2024-09-12,FL-619,PLAYCARD-STD,197,23.0,48


In [11]:
game_skus = [
    'BG-MONO',
    'GAME-MONO',
    'MONOPOLY-01',
    'CARDS-01',
    'PLAYCARD-STD'
]

game_inventory = inventory[
    inventory['sku'].isin(game_skus)
].copy()

periods = {
    'Baseline Sep 8-15': ('2024-09-08', '2024-09-15'),
    'Surge Sep 18-25': ('2024-09-18', '2024-09-25')
}

for period, (start, end) in periods.items():

    data = game_inventory[
        game_inventory['date'].between(start, end)
    ]

    print('\n', period)

    print(data.groupby('sku').agg(
        average_inventory=('units_on_hand', 'mean'),
        average_on_order=('units_on_order', 'mean')
    ))


 Baseline Sep 8-15
              average_inventory  average_on_order
sku                                              
BG-MONO               90.037037         52.227273
CARDS-01              94.555556         51.769231
GAME-MONO             86.541667         60.100000
MONOPOLY-01          131.900000         41.285714
PLAYCARD-STD         102.166667         39.875000

 Surge Sep 18-25
              average_inventory  average_on_order
sku                                              
BG-MONO               80.192308         51.619048
CARDS-01              79.178571         43.080000
GAME-MONO             62.960000         45.111111
MONOPOLY-01           70.350000         51.625000
PLAYCARD-STD          75.388889         54.142857


In [12]:
print(
    inventory.groupby('sku').agg(
        stores=('store_id', 'nunique'),
        average_stock=('units_on_hand', 'mean')
    ).sort_values('average_stock', ascending=False)
)

               stores  average_stock
sku                                 
MONOPOLY-01        12      88.945652
SOUP-CAN           12      88.737500
TOILETPAPER12      12      86.068966
BFORM-01           12      84.689320
BABYFORMULA        12      83.698113
TARP-HD            12      83.288889
BEER-24            12      82.127907
PTOWEL-6           12      81.869565
PLAYCARD-STD       12      81.534247
UNKNOWN-99         12      81.131868
SKU#459812         12      81.047059
CHIPS-VP           12      81.011364
BATT-100           12      81.000000
CNSOUP-01          12      80.127660
CARDS-01           12      80.072917
PNUTBTR-16         12      79.597222
FLASH-01           12      79.462500
BAT-AA4            12      79.384615
BATT_AA_4PK        12      79.106383
PORTCHARGER        12      79.089744
BREAD-WH           12      78.279070
CHIPVARIETY        12      78.214286
PT-12              12      78.171717
PAPERTOWEL6        12      78.155556
CANDLE-EMRG        12      78.154762
W